In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

In [2]:
train = pd.read_csv('../data/preprocessed/train.csv')
test  = pd.read_csv('../data/preprocessed/test.csv')

X = train.drop(columns=['Unnamed: 0', 'building_id', 'damage_grade'], errors='ignore')
y = train['damage_grade']
X_test = test.drop(columns=['Unnamed: 0', 'building_id', 'damage_grade'], errors='ignore')

print("Train:", X.shape, "| Test:", X_test.shape)

Train: (260601, 61) | Test: (86868, 61)


In [3]:
cat_cols = [
    'land_surface_condition', 'foundation_type', 'roof_type',
    'ground_floor_type', 'other_floor_type', 'position',
    'plan_configuration', 'legal_ownership_status'
]

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    X_test[col] = le.transform(X_test[col])

In [4]:
def target_encode_cv(X_tr, X_te, col, target_series, n_splits=5):
    global_mean = target_series.mean()
    encoded_train = np.zeros(len(X_tr))
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr_idx, val_idx in skf.split(X_tr, target_series):
        fold_map = target_series.iloc[tr_idx].groupby(X_tr[col].iloc[tr_idx]).mean()
        encoded_train[val_idx] = X_tr[col].iloc[val_idx].map(fold_map).fillna(global_mean)
    
    full_map = target_series.groupby(X_tr[col]).mean()
    encoded_test = X_te[col].map(full_map).fillna(global_mean)
    
    return encoded_train, encoded_test

for geo in ['geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id']:
    X[f'{geo}_te'], X_test[f'{geo}_te'] = target_encode_cv(X, X_test, geo, y)

print("Features after geo encoding:", X.shape[1])

Features after geo encoding: 64


In [5]:
for geo in ['geo_level_2_id', 'geo_level_3_id']:
    for stat_col in ['age', 'count_floors_pre_eq', 'area_percentage']:
        agg = y.groupby(X[geo]).mean().rename(f'{geo}_{stat_col}_mean')
        X = X.join(agg, on=geo)
        X_test = X_test.join(agg, on=geo)

print("Features after geo aggregations:", X.shape[1])

Features after geo aggregations: 70


In [ ]:
geo_cols = ['geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros((len(X_test), 5))
model_params = {
    "n_estimators": 2000,
    "learning_rate": 0.02,   
    "num_leaves": 255,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "random_state": 42,
    "verbose": -1
}

SMOOTH_K = 10  # smoothing factor for sparse regions

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr  = X.iloc[tr_idx].copy().reset_index(drop=True)
    X_val = X.iloc[val_idx].copy().reset_index(drop=True)
    X_te  = X_test.copy().reset_index(drop=True)
    y_tr  = y.iloc[tr_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)

    global_mean = y_tr.mean()

    for geo in geo_cols:
        agg = y_tr.groupby(X_tr[geo]).agg(['mean', 'count'])
        smoothed = (agg['mean'] * agg['count'] + global_mean * SMOOTH_K) / (agg['count'] + SMOOTH_K)

        X_tr[f'{geo}_te']  = X_tr[geo].map(smoothed).fillna(global_mean)
        X_val[f'{geo}_te'] = X_val[geo].map(smoothed).fillna(global_mean)
        X_te[f'{geo}_te']  = X_te[geo].map(smoothed).fillna(global_mean)

        for cls in [1, 2, 3]:
            binary = (y_tr == cls).astype(float)
            cls_agg = binary.groupby(X_tr[geo]).agg(['mean', 'count'])
            global_cls_mean = binary.mean()
            cls_smoothed = (cls_agg['mean'] * cls_agg['count'] + global_cls_mean * SMOOTH_K) / (cls_agg['count'] + SMOOTH_K)

            X_tr[f'{geo}_cls{cls}']  = X_tr[geo].map(cls_smoothed).fillna(global_cls_mean)
            X_val[f'{geo}_cls{cls}'] = X_val[geo].map(cls_smoothed).fillna(global_cls_mean)
            X_te[f'{geo}_cls{cls}']  = X_te[geo].map(cls_smoothed).fillna(global_cls_mean)

    model = lgb.LGBMClassifier(**model_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1)
        ]
    )

    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred
    test_preds[:, fold] = model.predict(X_te)

    print(f"Fold {fold+1} F1: {f1_score(y_val, val_pred, average='micro'):.4f} | best iter: {model.best_iteration_}")

print(f"\nOOF Micro F1: {f1_score(y, oof_preds, average='micro'):.4f}")

Fold 1 F1: 0.7484 | best iter: 309
Fold 2 F1: 0.7500 | best iter: 299
Fold 3 F1: 0.7503 | best iter: 302
Fold 4 F1: 0.7499 | best iter: 301
Fold 5 F1: 0.7481 | best iter: 299

OOF Micro F1: 0.7493


In [17]:
final_preds = np.round(test_preds.mean(axis=1)).astype(int).clip(1, 3)

submission = pd.DataFrame({
    'building_id': test['building_id'],
    'damage_grade': final_preds
})
submission.to_csv('../data/submission.csv', index=False)

In [ ]:
import joblib


# Save model
joblib.dump(final_model, '../models/lgbm_earthquake.pkl')


['../models/lgbm_earthquake.pkl']